# {{PROJECT_NAME}} — Modeling

**Owner(s):**  
**Goal:** Train candidate models, compare them, and select the champion.

Once the champion is chosen, update `CHAMPION_PARAMS` in `src/train.py` and document the decision in `docs/decisions.md`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
import joblib
from pathlib import Path

## 1. Load processed features

In [ ]:
X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_val   = pd.read_parquet('../data/processed/X_val.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet').squeeze()
y_val   = pd.read_parquet('../data/processed/y_val.parquet').squeeze()

print(f'Train: {X_train.shape} | Val: {X_val.shape}')
print(f'Positive rate — train: {y_train.mean():.2%} | val: {y_val.mean():.2%}')

## 2. Baseline (always predict majority class)

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_prob = baseline.predict_proba(X_val)[:, 1]
print(f'Baseline PR-AUC: {average_precision_score(y_val, baseline_prob):.4f}')

## 3. XGBoost experiments

TODO: try different hyperparameters and log the results in the table below.

In [ ]:
# TODO: tune these — start simple, add complexity only if needed
params = {
    'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 1,  # TODO: set to (neg / pos) for imbalanced data
    'eval_metric': 'aucpr',
    'early_stopping_rounds': 30,
    'random_state': 42,
    'n_jobs': -1,
}

model = XGBClassifier(**params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

val_prob = model.predict_proba(X_val)[:, 1]
print(f'\nPR-AUC:  {average_precision_score(y_val, val_prob):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_val, val_prob):.4f}')

## 4. Experiment log

Copy a row for each experiment you run.

| # | Key params | PR-AUC | ROC-AUC | Notes |
|---|---|---|---|---|
| 1 | default | TODO | TODO | baseline XGB |
| 2 | TODO | TODO | TODO | TODO |

## 5. Choose operating threshold

TODO: pick a threshold that balances precision and recall for {{PERSONA}}.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_val, val_prob)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall curve')
plt.grid(True)
plt.show()

# TODO: pick threshold and set OPERATING_THRESHOLD in .env
THRESHOLD = 0.5

## 6. Save champion model

In [ ]:
# Run this only once you've chosen the champion
Path('../models').mkdir(exist_ok=True)
joblib.dump(model, '../models/champion.pkl')
print('Champion model saved.')